In [1]:
!pip install -q opencv-python-headless numpy matplotlib scikit-learn shapely langchain-text-splitters
!pip install paddleocr
!pip install paddlepaddle

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.8/120.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/

# RUN

In [33]:
# ==============================
# INSTALL + IMPORT
# ==============================
import cv2
import numpy as np
import os
from paddleocr import PaddleOCR



In [34]:
# ==============================
# INIT OCR
# ==============================
os.environ['FLAGS_enable_pir_api'] = '0'

ocr = PaddleOCR(
    lang='vi',
    # use_textline_orientation=True,
    det_db_thresh=0.3,
    det_db_box_thresh=0.5,
    det_db_unclip_ratio=1.15,
    enable_mkldnn=False,
    use_angle_cls=True
)



/tmp/ipykernel_9324/1810117698.py:6: DeprecationWarning: The parameter `det_db_thresh` has been deprecated and will be removed in the future. Please use `text_det_thresh` instead.
  ocr = PaddleOCR(
/tmp/ipykernel_9324/1810117698.py:6: DeprecationWarning: The parameter `det_db_box_thresh` has been deprecated and will be removed in the future. Please use `text_det_box_thresh` instead.
  ocr = PaddleOCR(
/tmp/ipykernel_9324/1810117698.py:6: DeprecationWarning: The parameter `det_db_unclip_ratio` has been deprecated and will be removed in the future. Please use `text_det_unclip_ratio` instead.
  ocr = PaddleOCR(
/tmp/ipykernel_9324/1810117698.py:6: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  ocr = PaddleOCR(
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/offic

In [35]:
# ==============================
# PREPROCESS
# ==============================
def preprocess_image(path):
    img = cv2.imread(path)
    if img is None:
        raise ValueError("Image not loaded")

    # resize về kích thước hợp lý
    h, w = img.shape[:2]
    scale = 1400 / max(h, w)
    if scale < 1:
        img = cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    return img


In [36]:
# ==============================
# RECTIFY (FIX NGHIÊNG)
# ==============================
def rectify_document(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    edged = cv2.Canny(gray, 50, 150)

    contours, _ = cv2.findContours(edged, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)

    for c in contours[:5]:
        peri = cv2.arcLength(c, True)
        approx = cv2.approxPolyDP(c, 0.02 * peri, True)

        if len(approx) == 4:
            pts = approx.reshape(4,2)

            rect = np.zeros((4,2), dtype="float32")
            s = pts.sum(axis=1)
            rect[0] = pts[np.argmin(s)]
            rect[2] = pts[np.argmax(s)]

            diff = np.diff(pts, axis=1)
            rect[1] = pts[np.argmin(diff)]
            rect[3] = pts[np.argmax(diff)]

            (tl, tr, br, bl) = rect

            w = int(max(np.linalg.norm(br-bl), np.linalg.norm(tr-tl)))
            h = int(max(np.linalg.norm(tr-br), np.linalg.norm(tl-bl)))

            if w < 50 or h < 50:
                return img

            dst = np.array([[0,0],[w-1,0],[w-1,h-1],[0,h-1]], dtype="float32")
            M = cv2.getPerspectiveTransform(rect, dst)

            return cv2.warpPerspective(img, M, (w,h))

    return img


In [37]:
# ==============================
# OCR DETECT
# ==============================
def detect_text(image):
    lines = []

    # ===== try new API =====
    if hasattr(ocr, "predict"):
        result = ocr.predict(image)

        if isinstance(result, list) and len(result) > 0:
            res = result[0]

            polys = res.get('dt_polys', [])
            texts = res.get('rec_texts', [])
            scores = res.get('rec_scores', [])

            for i, poly in enumerate(polys):
                pts = np.array(poly, dtype=np.float32)
                if pts.ndim != 2 or pts.shape[1] != 2:
                    continue

                x0 = float(np.min(pts[:,0]))
                y0 = float(np.min(pts[:,1]))
                x1 = float(np.max(pts[:,0]))
                y1 = float(np.max(pts[:,1]))

                text = texts[i] if i < len(texts) else ""
                score = scores[i] if i < len(scores) else 1.0

                lines.append([x0,y0,x1,y1,text,score,pts])

    else:
        # ===== fallback API =====
        result = ocr.ocr(image)

        if isinstance(result, list):
            for block in result:
                if block is None:
                    continue

                for item in block:
                    if not isinstance(item, (list, tuple)):
                        continue

                    box = item[0]

                    if not isinstance(box, (list, tuple)):
                        continue

                    pts = np.array(box)

                    if pts.ndim != 2 or pts.shape[1] != 2:
                        continue

                    x0 = float(np.min(pts[:,0]))
                    y0 = float(np.min(pts[:,1]))
                    x1 = float(np.max(pts[:,0]))
                    y1 = float(np.max(pts[:,1]))

                    text = ""
                    score = 1.0

                    if len(item) > 1:
                        val = item[1]
                        if isinstance(val, (list, tuple)):
                            text = val[0]
                            if len(val) > 1:
                                score = val[1]
                        elif isinstance(val, str):
                            text = val

                    lines.append([x0,y0,x1,y1,text,score,pts])

    return [item[6] for item in lines]




In [38]:
# ==============================
# CROP BY TEXT
# ==============================
def crop_by_text(polys, image):
    if not polys:
        return image

    xs, ys = [], []

    for poly in polys:
        pts = np.array(poly)

        xs += list(pts[:,0])
        ys += list(pts[:,1])

    x0, y0 = int(min(xs)), int(min(ys))
    x1, y1 = int(max(xs)), int(max(ys))

    pad = 20
    h, w = image.shape[:2]

    return image[
        max(0,y0-pad):min(h,y1+pad),
        max(0,x0-pad):min(w,x1+pad)
    ]

In [39]:
# ==============================
# UTIL
# ==============================
def polygon_to_bbox(poly):
    pts = np.array(poly).reshape(-1, 2)   # 🔥 ép về numpy trước

    x0 = float(np.min(pts[:,0]))
    y0 = float(np.min(pts[:,1]))
    x1 = float(np.max(pts[:,0]))
    y1 = float(np.max(pts[:,1]))

    return [x0,y0,x1,y1]

def shrink_poly(poly, scale=0.9):
    pts = np.array(poly, dtype=np.float32)
    center = np.mean(pts, axis=0)
    pts = center + (pts - center) * scale
    return pts

def crop_polygon(img, poly):
    # 🔥 shrink trực tiếp ở đây
    pts = np.array(shrink_poly(poly, 0.9), dtype="float32")

    # Sắp xếp tọa độ: top-left, top-right, bottom-right, bottom-left
    rect = np.zeros((4, 2), dtype="float32")

    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]

    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]

    (tl, tr, br, bl) = rect

    width = int(max(np.linalg.norm(br - bl), np.linalg.norm(tr - tl)))
    height = int(max(np.linalg.norm(tr - br), np.linalg.norm(tl - bl)))

    if width <= 0 or height <= 0:
        return None

    dst = np.array([
        [0, 0],
        [width - 1, 0],
        [width - 1, height - 1],
        [0, height - 1]
    ], dtype="float32")

    M = cv2.getPerspectiveTransform(rect, dst)
    warped = cv2.warpPerspective(img, M, (width, height))

    return warped



In [40]:
# ==============================
# GROUP LINE (STABLE)
# ==============================
def group_lines(polygons):
    if not polygons:
        return []

    bboxes = [polygon_to_bbox(p) for p in polygons]
    bboxes = sorted(bboxes, key=lambda b: b[1])

    lines = []
    current = [bboxes[0]]

    for box in bboxes[1:]:
        prev = current[-1]

        if abs(box[1] - prev[1]) < 0.6 * (prev[3]-prev[1]):
            current.append(box)
        else:
            lines.append(current)
            current = [box]

    lines.append(current)

    merged = []
    for line in lines:
        xs = [b[0] for b in line] + [b[2] for b in line]
        ys = [b[1] for b in line] + [b[3] for b in line]

        merged.append([
            min(xs), min(ys),
            max(xs), max(ys)
        ])

    return merged




In [41]:
# ==============================
# FIX BOX
# ==============================

def split_wide_boxes(lines):
    new_lines = []

    for x0,y0,x1,y1 in lines:
        if x1 <= x0 or y1 <= y0:
            continue  # skip lỗi

        width = x1 - x0
        height = y1 - y0

        if width > 3 * height:  # chỉ split khi thật sự dài
            mid = (y0 + y1) // 2

            if mid > y0 and mid < y1:
                new_lines.append([x0, y0, x1, mid])
                new_lines.append([x0, mid, x1, y1])
            else:
                new_lines.append([x0,y0,x1,y1])
        else:
            new_lines.append([x0,y0,x1,y1])

    return new_lines

In [42]:
# ==============================
# EXPAND BOX (NHẸ)
# ==============================
def expand_box(box, shape):
    x0,y0,x1,y1 = box
    H,W = shape[:2]

    bw = x1 - x0
    bh = y1 - y0

    pad_x = int(0.05 * bw)
    pad_y = int(0.15 * bh)

    x0 = max(0, x0 - pad_x)
    y0 = max(0, y0 - pad_y)
    x1 = min(W, x1 + pad_x)
    y1 = min(H, y1 + pad_y)

    return [x0,y0,x1,y1]



In [43]:
# ==============================
# DRAW
# ==============================
def draw_lines(image, polys):
    out = image.copy()
    for pts in polys:
        pts = shrink_poly(pts, 0.9)
        pts = pts.astype(np.int32).reshape((-1, 1, 2))
        # Vẽ viền đỏ mảnh (khít và sang hơn)
        cv2.polylines(out, [pts], True, (0, 0, 255), 1, cv2.LINE_AA)
    return out




In [44]:
def process_image(path, output_dir):
    name = os.path.basename(path).split('.')[0]

    # 1. Tiền xử lý
    img = preprocess_image(path)

    # ❌ KHÔNG warp nữa (rất quan trọng)
    img = rectify_document(img)

    # 2. OCR
    polys = detect_text(img)

    if len(polys) == 0:
        print(f"❌ {name} | OCR FAIL")
        return

    print(f"{name} | Tìm thấy {len(polys)} vùng văn bản")

    # 3. Visualize (vẽ trực tiếp trên ảnh OCR)
    vis = draw_lines(img, polys)

    # 4. Save
    os.makedirs(output_dir, exist_ok=True)
    cv2.imwrite(f"{output_dir}/{name}_vis.jpg", vis)

    # 5. Crop từng dòng
    for i, poly in enumerate(polys):
        crop = crop_polygon(img, poly)

        if crop is None or crop.size == 0:
            continue

        cv2.imwrite(f"{output_dir}/{name}_line_{i}.jpg", crop)

    print("✅ Done:", name)

In [45]:
# ==============================
# BATCH
# ==============================
def process_folder(input_dir, output_dir):
    files = [f for f in os.listdir(input_dir)
             if f.lower().endswith(('.jpg','.png','.jpeg'))]

    for i,f in enumerate(files):
        print(f"[{i+1}/{len(files)}] {f}")
        process_image(os.path.join(input_dir, f), output_dir)


# ==============================
# RUN (COLAB)
# ==============================
from google.colab import drive
drive.mount('/content/drive')

input_dir = "/content/drive/MyDrive/OCR_INPUT"
output_dir = "/content/drive/MyDrive/OCR_OUTPUT"

process_folder(input_dir, output_dir)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[1/26] mcocr_public_145013atlmq.jpg
mcocr_public_145013atlmq | Tìm thấy 49 vùng văn bản
✅ Done: mcocr_public_145013atlmq
[2/26] mcocr_public_145013atnil.jpg
mcocr_public_145013atnil | Tìm thấy 24 vùng văn bản
✅ Done: mcocr_public_145013atnil
[3/26] mcocr_public_145013aukcu.jpg
mcocr_public_145013aukcu | Tìm thấy 20 vùng văn bản
✅ Done: mcocr_public_145013aukcu
[4/26] mcocr_public_145013awcbh.jpg
mcocr_public_145013awcbh | Tìm thấy 37 vùng văn bản
✅ Done: mcocr_public_145013awcbh
[5/26] mcocr_public_145013bchvc.jpg
mcocr_public_145013bchvc | Tìm thấy 41 vùng văn bản
✅ Done: mcocr_public_145013bchvc
[6/26] mcocr_public_145013azaup.jpg
mcocr_public_145013azaup | Tìm thấy 27 vùng văn bản
✅ Done: mcocr_public_145013azaup
[7/26] mcocr_public_145013bajyy.jpg
mcocr_public_145013bajyy | Tìm thấy 44 vùng văn bản
✅ Done: mcocr_public_145013bajyy
[8/26] mcocr_public_1450

# RUN (OCR CHUẨN)

In [ ]:
# # ==============================
# # IMPORT
# # ==============================
# import cv2
# import numpy as np
# import os
# from paddleocr import PaddleOCR

# # ==============================
# # INIT OCR (VERSION MỚI)
# # ==============================
# os.environ['FLAGS_enable_pir_api'] = '0'

# ocr = PaddleOCR(
#     lang='vi',
#     use_textline_orientation=True,

#     text_det_thresh=0.25,
#     text_det_box_thresh=0.45,
#     text_det_unclip_ratio=1.6,

#     enable_mkldnn=False
# )

# # ==============================
# # PREPROCESS
# # ==============================
# def preprocess_image(path):
#     img = cv2.imread(path)
#     if img is None:
#         raise ValueError("Image not loaded")

#     h, w = img.shape[:2]
#     scale = 1400 / max(h, w)
#     if scale < 1:
#         img = cv2.resize(img, None, fx=scale, fy=scale)

#     return img


# # ==============================
# # OCR (AUTO COMPATIBLE)
# # ==============================
# def detect_lines(image):
#     lines = []

#     # ===== try new API =====
#     if hasattr(ocr, "predict"):
#         result = ocr.predict(image)

#         if isinstance(result, list) and len(result) > 0:
#             res = result[0]

#             polys = res.get('dt_polys', [])
#             texts = res.get('rec_texts', [])
#             scores = res.get('rec_scores', [])

#             for i, poly in enumerate(polys):
#                 pts = np.array(poly)

#                 if pts.ndim != 2 or pts.shape[1] != 2:
#                     continue

#                 x0 = float(np.min(pts[:,0]))
#                 y0 = float(np.min(pts[:,1]))
#                 x1 = float(np.max(pts[:,0]))
#                 y1 = float(np.max(pts[:,1]))

#                 text = texts[i] if i < len(texts) else ""
#                 score = scores[i] if i < len(scores) else 1.0

#                 lines.append([x0,y0,x1,y1,text,score,pts])

#     else:
#         # ===== fallback API =====
#         result = ocr.ocr(image)

#         if isinstance(result, list):
#             for block in result:
#                 if block is None:
#                     continue

#                 for item in block:
#                     if not isinstance(item, (list, tuple)):
#                         continue

#                     box = item[0]

#                     if not isinstance(box, (list, tuple)):
#                         continue

#                     pts = np.array(box)

#                     if pts.ndim != 2 or pts.shape[1] != 2:
#                         continue

#                     x0 = float(np.min(pts[:,0]))
#                     y0 = float(np.min(pts[:,1]))
#                     x1 = float(np.max(pts[:,0]))
#                     y1 = float(np.max(pts[:,1]))

#                     text = ""
#                     score = 1.0

#                     if len(item) > 1:
#                         val = item[1]
#                         if isinstance(val, (list, tuple)):
#                             text = val[0]
#                             if len(val) > 1:
#                                 score = val[1]
#                         elif isinstance(val, str):
#                             text = val

#                     lines.append([x0,y0,x1,y1,text,score,pts])

#     return lines


# # ==============================
# # CROP THEO TEXT (KHÔNG MẤT CHỮ)
# # ==============================
# def crop_by_text(lines, image):
#     if not lines:
#         return image

#     xs, ys = [], []

#     for x0,y0,x1,y1,_,_,_ in lines:
#         xs += [x0,x1]
#         ys += [y0,y1]

#     x0, y0 = int(min(xs)), int(min(ys))
#     x1, y1 = int(max(xs)), int(max(ys))

#     pad = 20
#     h,w = image.shape[:2]

#     return image[
#         max(0,y0-pad):min(h,y1+pad),
#         max(0,x0-pad):min(w,x1+pad)
#     ]


# # ==============================
# # RECTIFY (OPTIONAL)
# # ==============================
# def rectify_document(img):
#     gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
#     edged = cv2.Canny(gray, 50, 150)

#     contours, _ = cv2.findContours(edged, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
#     contours = sorted(contours, key=cv2.contourArea, reverse=True)

#     for c in contours[:5]:
#         peri = cv2.arcLength(c, True)
#         approx = cv2.approxPolyDP(c, 0.02 * peri, True)

#         if len(approx) == 4:
#             pts = approx.reshape(4,2).astype("float32")

#             rect = np.zeros((4,2), dtype="float32")
#             s = pts.sum(axis=1)
#             rect[0] = pts[np.argmin(s)]
#             rect[2] = pts[np.argmax(s)]

#             diff = np.diff(pts, axis=1)
#             rect[1] = pts[np.argmin(diff)]
#             rect[3] = pts[np.argmax(diff)]

#             (tl, tr, br, bl) = rect

#             w = int(max(np.linalg.norm(br-bl), np.linalg.norm(tr-tl)))
#             h = int(max(np.linalg.norm(tr-br), np.linalg.norm(tl-bl)))

#             if w < 50 or h < 50:
#                 return img

#             dst = np.array([[0,0],[w-1,0],[w-1,h-1],[0,h-1]], dtype="float32")
#             M = cv2.getPerspectiveTransform(rect, dst)

#             return cv2.warpPerspective(img, M, (w,h))

#     return img


# # ==============================
# # DRAW (POLYGON CHUẨN)
# # ==============================
# def draw_lines(image, lines):
#     out = image.copy()

#     for i,(x0,y0,x1,y1,text,score,pts) in enumerate(lines):

#         if score < 0.5:
#             continue

#         pts_int = pts.astype(np.int32)

#         cv2.polylines(out, [pts_int], True, (0,255,0), 2)

#         cv2.putText(out,
#                     text[:25],
#                     (int(x0), int(y0)-5),
#                     cv2.FONT_HERSHEY_SIMPLEX,
#                     0.5,
#                     (0,255,0),
#                     1)

#     return out


# # ==============================
# # EXPAND BOX (AN TOÀN)
# # ==============================
# def expand_box(box, shape):
#     x0,y0,x1,y1 = box
#     H,W = shape[:2]

#     bw = x1 - x0
#     bh = y1 - y0

#     pad_x = int(0.1 * bw)
#     pad_y = int(0.3 * bh)

#     x0 = max(0, x0 - pad_x)
#     y0 = max(0, y0 - pad_y)
#     x1 = min(W, x1 + pad_x)
#     y1 = min(H, y1 + pad_y)

#     return [x0,y0,x1,y1]


# # ==============================
# # PROCESS IMAGE (FIX LỆCH BOX)
# # ==============================
# def process_image(path, output_dir):
#     name = os.path.basename(path).split('.')[0]

#     img = preprocess_image(path)

#     # OCR lần 1
#     lines = detect_lines(img)
#     if not lines:
#         print("❌ OCR FAIL (raw)")
#         return

#     # crop
#     img = crop_by_text(lines, img)

#     # OCR lại (rất quan trọng)
#     lines = detect_lines(img)

#     # rectify (optional)
#     img = rectify_document(img)

#     # OCR lại lần nữa (fix lệch)
#     lines = detect_lines(img)

#     print(f"{name} | lines:", len(lines))

#     if not lines:
#         print("❌ OCR FAIL (final)")
#         return

#     os.makedirs(output_dir, exist_ok=True)

#     # visualize
#     vis = draw_lines(img, lines)
#     cv2.imwrite(f"{output_dir}/{name}_vis.jpg", vis)

#     # crop từng dòng
#     h, w = img.shape[:2]

#     for i,(x0,y0,x1,y1,text,score,pts) in enumerate(lines):

#         if score < 0.5:
#             continue

#         x0,y0,x1,y1 = expand_box([x0,y0,x1,y1], img.shape)

#         x0 = max(0, min(w-1, int(x0)))
#         x1 = max(0, min(w-1, int(x1)))
#         y0 = max(0, min(h-1, int(y0)))
#         y1 = max(0, min(h-1, int(y1)))

#         if x1 <= x0 or y1 <= y0:
#             continue

#         crop = img[y0:y1, x0:x1]

#         if crop is None or crop.size == 0:
#             continue

#         cv2.imwrite(f"{output_dir}/{name}_line_{i}.jpg", crop)

#     print("✅ Done:", name)


# # ==============================
# # PROCESS FOLDER
# # ==============================
# def process_folder(input_dir, output_dir):
#     files = [f for f in os.listdir(input_dir)
#              if f.lower().endswith(('.jpg','.png','.jpeg'))]

#     for i,f in enumerate(files):
#         print(f"[{i+1}/{len(files)}] {f}")
#         process_image(os.path.join(input_dir, f), output_dir)


# # ==============================
# # RUN (COLAB)
# # ==============================
# from google.colab import drive
# drive.mount('/content/drive')

# input_dir = "/content/drive/MyDrive/OCR_INPUT"
# output_dir = "/content/drive/MyDrive/OCR_OUTPUT"

# process_folder(input_dir, output_dir)

Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_server_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv5_server_det`.
Creating model: ('latin_PP-OCRv5_mobile_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_mod

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[1/26] mcocr_public_145013atlmq.jpg
mcocr_public_145013atlmq | lines: 49
✅ Done: mcocr_public_145013atlmq
[2/26] mcocr_public_145013atnil.jpg
mcocr_public_145013atnil | lines: 25
✅ Done: mcocr_public_145013atnil
[3/26] mcocr_public_145013aukcu.jpg
mcocr_public_145013aukcu | lines: 20
✅ Done: mcocr_public_145013aukcu
[4/26] mcocr_public_145013awcbh.jpg
mcocr_public_145013awcbh | lines: 14
✅ Done: mcocr_public_145013awcbh
[5/26] mcocr_public_145013bchvc.jpg
mcocr_public_145013bchvc | lines: 41
✅ Done: mcocr_public_145013bchvc
[6/26] mcocr_public_145013azaup.jpg
mcocr_public_145013azaup | lines: 28
✅ Done: mcocr_public_145013azaup
[7/26] mcocr_public_145013bajyy.jpg
mcocr_public_145013bajyy | lines: 43
✅ Done: mcocr_public_145013bajyy
[8/26] mcocr_public_145013avtwc.jpg
mcocr_public_145013avtwc | lines: 117
✅ Done: mcocr_public_145013avtwc
[9/26] mcocr_public_14